> **Companion notebook** — generated from `modules/13-laplace-em-vi.md` (the canonical, harness-verified text; regenerate with `python tools/make_notebooks.py`). Run cells top-to-bottom from the `curriculum/` directory so `figures/...` paths resolve. Cells marked *illustration only* are intentionally not executable.

# 13. Deterministic Approximations: One Idea (the ELBO) Three Ways

> **Spine.** Approximate the posterior with a computable shape; every shape has a signed, predictable failure — mean-field gets the mean right and the variance too small, by the factor $1-\rho^2$ exactly.
> **Which line?** Line 2, for when the line-2 normalizing integral is intractable and you refuse to sample: replace $p(\theta\mid x)$ with the closest member of a family you *can* compute. Laplace, EM, and variational inference are three choices of that family.
> **Promise.** After this module you can state the exact identity that every one of these methods optimizes, read off *which direction* each one errs, predict the mean-field variance deficit before running anything, and choose between a Gaussian-at-the-mode, an EM point estimate, and a full variational fit knowing what each throws away.
> **Prereqs.** Modules 03 (entropy/KL, defined once; the mode-seeking vs mode-covering promise), 05 (the Beta and NIG conjugate posteriors we approximate and grade against), 09 (Monte Carlo, for the SVI-vs-NUTS comparison), 12 (NUTS as the accuracy reference). **Runtime.** ~30 s (includes JAX JIT + NUTS).
> **Sources.** Booklet ch. 12–13; Bishop *PRML* ch. 10 by concept; BDA3 ch. 13 by concept.

Sampling (modules 09–12) attacks the intractable posterior by *drawing from it*. This module does the opposite: it replaces the posterior with a tractable stand-in $q(\theta)$ and pays for the substitution in a currency we already minted — bits of KL (module 03). Three named procedures — the **Laplace approximation**, the **EM algorithm**, and **variational inference** — look unrelated in most textbooks. They are one idea seen three ways: *pick a family for $q$, then make $q$ as close to the posterior as that family allows.* The single quantity that makes "as close as allows" precise is the **evidence lower bound (ELBO)**, and its defining identity is the spine of the whole module.

In [ ]:
# --- setup ---
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

SLUG = "13-laplace-em-vi"          # this module's figure dir
FIG = Path("figures") / SLUG
FIG.mkdir(parents=True, exist_ok=True)
SEED = 0
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.figsize": (7, 4), "figure.dpi": 110, "savefig.dpi": 150,
    "savefig.bbox": "tight", "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11,
})

def save(fig, name):
    out = FIG / f"{name}.png"
    fig.savefig(out)
    plt.close(fig)
    print(f"[fig] {out}")

## 13.1 The identity every method optimizes

Fix data $x$ and a model $p(x,\theta) = p(x\mid\theta)\,p(\theta)$. The thing you want is the posterior $p(\theta\mid x) = p(x,\theta)/p(x)$, and the thing that makes it hard is the **evidence** (marginal likelihood) $p(x)=\int p(x,\theta)\,d\theta$ in the denominator. Introduce *any* density $q(\theta)$ over the same unknowns and define the **ELBO**

$$\mathrm{ELBO}(q) \;=\; \mathbb{E}_{q}\!\big[\log p(x,\theta) - \log q(\theta)\big] \;=\; \underbrace{\mathbb{E}_q[\log p(x,\theta)]}_{\text{energy}} \;+\; \underbrace{H(q)}_{\text{entropy}}.$$

The name is earned by an exact identity. Write the evidence, insert $q/q$, and split the log:

$$\log p(x) \;=\; \mathbb{E}_q\!\Big[\log\tfrac{p(x,\theta)}{p(\theta\mid x)}\Big] \;=\; \mathbb{E}_q\!\Big[\log\tfrac{p(x,\theta)}{q(\theta)}\Big] + \mathbb{E}_q\!\Big[\log\tfrac{q(\theta)}{p(\theta\mid x)}\Big] \;=\; \mathrm{ELBO}(q) \;+\; \mathrm{KL}\!\big(q \,\|\, p(\cdot\mid x)\big).$$

$$\boxed{\;\log p(x) \;=\; \mathrm{ELBO}(q) \;+\; \mathrm{KL}\big(q \,\|\, p(\cdot\mid x)\big)\;}$$

The first equality uses that $\log p(x)$ does not depend on $\theta$, so $\mathbb{E}_q[\log p(x)] = \log p(x)$; the last line reads off the ELBO and a KL divergence. **The KL is to the posterior, not the prior** — this is the whole point, and a common misstatement. Because $\log p(x)$ is a constant (it does not involve $q$), the identity says $\mathrm{ELBO}(q)$ and $-\mathrm{KL}(q\,\|\,p(\cdot\mid x))$ move in lockstep: **maximizing the ELBO over $q$ is identically minimizing the KL from $q$ to the posterior.** You never need the intractable $p(x)$ to do it — the ELBO is computable from the joint $p(x,\theta)$ and your choice of $q$ alone.

A second derivation makes the "lower bound" literal. Since $\mathrm{KL}\ge 0$ (Gibbs' inequality, module 03), $\mathrm{ELBO}(q) \le \log p(x)$ for *every* $q$, with equality iff $q = p(\cdot\mid x)$. This is Jensen's inequality wearing work clothes: $\log p(x) = \log \mathbb{E}_q[p(x,\theta)/q(\theta)] \ge \mathbb{E}_q[\log p(x,\theta)/q(\theta)] = \mathrm{ELBO}(q)$, and the slack in Jensen *is* the KL. The evidence sits above every ELBO; the gap you leave open is exactly the bits by which $q$ misses the posterior.

That reframes all three methods as one sentence with three fillings: **choose a family $\mathcal{Q}$ for $q$, then maximize the ELBO within it.** Laplace takes $\mathcal{Q} = $ Gaussians — with one honest seam: it picks its Gaussian by matching the posterior's peak *curvature*, not by maximizing the ELBO over Gaussians, and the two choices coincide only as $n\to\infty$ (BvM), which is why §13.2 grades Laplace by its KL to the posterior directly; EM takes $\mathcal{Q}$ with a point mass on the parameters and the exact conditional on the latents; variational inference takes $\mathcal{Q} = $ a factorized ("mean-field") family and grinds the ELBO uphill by coordinate ascent. Different $\mathcal{Q}$, same objective, different signed failure.

Verify the identity where we can compute all three terms exactly. Take the coin of module 00: prior $\mathrm{Beta}(1,1)$, data 7 heads in 10, posterior $\mathrm{Beta}(8,4)$. The evidence is the Beta-Binomial normalizing constant in closed form; the ELBO and KL are one-dimensional integrals we do on a grid, for three candidate $q$'s.

In [ ]:
# 13.1 -- ELBO + KL = log p(x), verified for three candidate q's.
from scipy.special import gammaln, comb, logsumexp
a0, b0, s, f = 1.0, 1.0, 7, 3            # Beta(1,1) prior, 7 successes / 3 failures
n = s + f
an, bn = a0 + s, b0 + f                   # exact posterior Beta(8,4)

def logBeta(a, b):
    return gammaln(a) + gammaln(b) - gammaln(a + b)

# exact evidence: p(x) = C(n,s) B(a0+s, b0+f) / B(a0,b0)
logpx = np.log(comb(n, s)) + logBeta(an, bn) - logBeta(a0, b0)

th = np.linspace(1e-4, 1 - 1e-4, 200_001)
dth = th[1] - th[0]
logjoint = np.log(comb(n, s)) + s * np.log(th) + f * np.log(1 - th)   # log p(x, theta)
logpost = stats.beta(an, bn).logpdf(th)                               # log p(theta | x)

def elbo_and_kl(qa, qb):
    logq = stats.beta(qa, qb).logpdf(th)
    q = np.exp(logq)
    elbo = np.sum(q * (logjoint - logq)) * dth          # E_q[log p(x,theta) - log q]
    kl = np.sum(q * (logq - logpost)) * dth             # KL(q || posterior)
    return elbo, kl

print(f"exact log p(x)                = {logpx:.4f}")
for (qa, qb) in [(8, 4), (5, 5), (3, 9)]:
    elbo, kl = elbo_and_kl(qa, qb)
    tag = "  <- q = exact posterior" if (qa, qb) == (8, 4) else ""
    print(f"q=Beta({qa},{qb}): ELBO={elbo:8.4f}  KL={kl:.4f}  ELBO+KL={elbo + kl:.4f}{tag}")

The evidence is $\log p(x) = $ `-2.3979`. Setting $q$ to the true posterior $\mathrm{Beta}(8,4)$ gives $\mathrm{KL}=$ `0.0000` and $\mathrm{ELBO}=$ `-2.3979` — the bound is tight, as promised. A moderately wrong $q=\mathrm{Beta}(5,5)$ scores $\mathrm{ELBO}=$ `-3.1495` and $\mathrm{KL}=$ `0.7516`; a badly-placed $q=\mathrm{Beta}(3,9)$ scores $\mathrm{ELBO}=$ `-7.5064` and $\mathrm{KL}=$ `5.1085`. In every row $\mathrm{ELBO}+\mathrm{KL}=$ `-2.3979` to four decimals: the identity holds exactly, and a lower KL is a higher ELBO, one-for-one.

In [ ]:
# figure: ELBO tracks log p(x) minus the KL gap, across a one-parameter q family.
cc = np.linspace(0.15, 6.0, 120)                 # concentration multiplier
elbos = np.array([elbo_and_kl(c * an, c * bn)[0] for c in cc])   # Beta(c*an, c*bn): mean fixed
kls = logpx - elbos
fig, ax = plt.subplots()
ax.axhline(logpx, color="k", ls="--", lw=1.5, label=r"$\log p(x)$ (evidence)")
ax.plot(cc, elbos, color="C1", lw=2, label=r"ELBO$(q_c)$")
c_star = cc[np.argmax(elbos)]
ax.axvline(1.0, color="C0", ls=":", lw=1)
ax.fill_between(cc, elbos, logpx, color="C3", alpha=0.15)
ax.annotate("KL gap", (3.6, (logpx + elbos[np.argmin(abs(cc - 3.6))]) / 2),
            color="C3", fontsize=10)
ax.annotate("ELBO touches evidence\nwhen q = posterior (c=1)", (1.05, logpx),
            textcoords="offset points", xytext=(10, -34), fontsize=9,
            arrowprops=dict(arrowstyle="->"))
ax.set(xlabel=r"concentration $c$ in $q_c=\mathrm{Beta}(c\,a_n,\,c\,b_n)$",
       ylabel="nats", ylim=(logpx - 3, logpx + 0.4),
       title="Maximizing the ELBO = closing the KL gap to the evidence")
ax.legend()
save(fig, "elbo_identity")

![The horizontal evidence line log p(x) with the ELBO curve rising to touch it exactly at c=1 (where q equals the posterior) and falling away on either side; the shaded vertical gap between them is the KL divergence.](figures/13-laplace-em-vi/elbo_identity.png)

The figure sweeps a family $q_c=\mathrm{Beta}(c\,a_n, c\,b_n)$ whose mean is pinned to the posterior mean but whose concentration $c$ varies. The ELBO curve rises to *kiss* the evidence line exactly at $c=1$ (the true posterior) and drops away on both sides; the vertical distance to the evidence line is the KL. Everything below is a fight to close that gap inside a restricted family.

## 13.2 Laplace: a Gaussian at the peak — on the right scale

The cheapest $q$ is a Gaussian centered at the posterior mode $\theta^\star$ (the MAP) with covariance set by the curvature there: $\Sigma^{-1} = -\nabla^2 \log p(\theta\mid x)\big|_{\theta^\star}$, the negative Hessian of the log-posterior. This is a second-order Taylor expansion of $\log p(\theta\mid x)$ around its peak — exact if the posterior is Gaussian, good when it is nearly so (which, by the Bernstein–von Mises theorem of module 08, it becomes as $n\to\infty$). It costs one optimization and one Hessian, no integration.

But *on which coordinates?* A Beta posterior lives on $[0,1]$; a Gaussian lives on $\mathbb{R}$ and will spill probability past the boundary. Run Laplace on the raw $\theta$ scale for a $\mathrm{Beta}(2,5)$ posterior (mode $0.2$) and the fitted Gaussian assigns `0.1318` of its mass to $\theta<0$ — impossible conversion rates. The fix is the same one deep learning uses when it parameterizes a variance as $\log\sigma$: **work in an unconstrained coordinate.** Map $\theta\mapsto\phi=\mathrm{logit}(\theta)=\log\frac{\theta}{1-\theta}\in\mathbb{R}$, Laplace-approximate there, and transform back. Comparing raw-scale Laplace to the exact posterior would be grading it on a rigged track; the logit scale is the fair one.

The transformed posterior is clean. With a Jacobian $d\theta/d\phi = \theta(1-\theta)$, a $\mathrm{Beta}(\alpha,\beta)$ density in $\theta$ becomes, in $\phi$, proportional to $\theta^{\alpha}(1-\theta)^{\beta}$ with $\theta=\sigma(\phi)$. Its log has derivative $\alpha - (\alpha+\beta)\theta$, so the mode sits at $\theta^\star = \alpha/(\alpha+\beta)$ — the Beta *mean*, mapped through logit — and the second derivative is $-(\alpha+\beta)\theta(1-\theta)$, giving Laplace variance $(\alpha+\beta)/(\alpha\beta)$ in $\phi$. Grade the fit by $\mathrm{KL}(q\,\|\,\text{posterior})$ in $\phi$-space, at two sample sizes.

In [ ]:
# 13.2 -- Laplace on the logit scale for a Beta posterior; KL to exact at n=5 and n=200.
phi = np.linspace(-14, 14, 400_001)
dphi = phi[1] - phi[0]
thv = 1.0 / (1.0 + np.exp(-phi))                     # theta = sigmoid(phi)

def laplace_logit(alpha, beta):
    """Laplace fit of Beta(alpha,beta) on the logit scale; returns (mean, var, KL-to-exact)."""
    log_exact = alpha * np.log(thv) + beta * np.log(1 - thv)        # logit-Beta, unnormalized
    log_exact -= logsumexp(log_exact + np.log(dphi))               # normalize on the grid
    mu = np.log(alpha / beta)                                       # logit(alpha/(alpha+beta))
    var = (alpha + beta) / (alpha * beta)                          # 1 / curvature at the mode
    log_lap = stats.norm(mu, np.sqrt(var)).logpdf(phi)
    kl = np.sum(np.exp(log_lap) * (log_lap - log_exact)) * dphi     # KL(Laplace || exact)
    return mu, var, kl

# Prior Beta(1,1), true theta = 0.2. n=5 -> 1 success; n=200 -> 40 successes.
fits = {}
for nn, ss in [(5, 1), (200, 40)]:
    alpha, beta = 1 + ss, 1 + (nn - ss)
    mu, var, kl = laplace_logit(alpha, beta)
    fits[nn] = (alpha, beta, mu, var, kl)
    print(f"n={nn:>3}: posterior Beta({alpha},{beta})  logit-Laplace N({mu:.3f}, {var:.4f})"
          f"  KL(Lap||post) = {kl:.4f}")
print(f"small-n penalty: KL ratio n=5 / n=200 = {fits[5][4] / fits[200][4]:.1f}x")

# raw-scale Laplace leaks mass past the boundary (the unfair comparison we avoid):
mode_raw, var_raw = 0.2, 1.0 / ((2 - 1) / 0.2**2 + (5 - 1) / 0.8**2)   # Beta(2,5) on raw theta
print(f"raw-scale Laplace Beta(2,5): P(theta<0) = {stats.norm(mode_raw, np.sqrt(var_raw)).cdf(0):.4f}"
      f"  (impossible mass the logit scale forbids)")

**Setup.** Laplace is "just a Gaussian approximation." **Predict.** You will fit it on the logit scale — the fair one — at $n=5$ and $n=200$, and grade both by KL. Commit to a number before reading on: the $n=5$ KL will be within **2×**, **5×**, or **10×** of the $n=200$ KL — pick one. **Reason.** The naive read is that logit-space linearizes the constraint away, so small $n$ should already be nearly as good: 2×, maybe 5×.

**Reconcile.** At $n=200$ the fit is excellent: $\mathrm{KL}=$ `0.0022` nats, the logit-posterior is all but Gaussian (BvM has kicked in). At $n=5$ the same construction scores $\mathrm{KL}=$ `0.0264` nats — `11.8`× worse, past the top of the menu. Five observations leave the logit-posterior visibly skewed, and a symmetric Gaussian cannot track a skew; Laplace *degrades at small $n$*, precisely where you most want a cheap posterior and precisely where the CLT that justifies it has not arrived. The logit reparameterization bought fairness (no impossible mass — contrast the `0.1318` the raw scale wastes below zero) but not a miracle.

In [ ]:
# figure: logit-Laplace transformed back to theta, at n=5 (skew missed) and n=200 (nailed).
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
tg = np.linspace(1e-3, 1 - 1e-3, 400)
phig = np.log(tg / (1 - tg))
for k, nn in enumerate((5, 200)):
    alpha, beta, mu, var, kl = fits[nn]
    ax[k].plot(tg, stats.beta(alpha, beta).pdf(tg), "k--", lw=2, label="exact posterior")
    # push N(mu,var) in phi through theta=sigmoid(phi): density * |dphi/dtheta| = /(theta(1-theta))
    lap_theta = stats.norm(mu, np.sqrt(var)).pdf(phig) / (tg * (1 - tg))
    ax[k].plot(tg, lap_theta, color="C1", lw=2, label="logit-Laplace")
    ax[k].set(title=f"n={nn}: KL = {kl:.4f} nats", xlabel=r"$\theta$", ylabel="density")
    ax[k].legend()
fig.suptitle("Laplace on the logit scale: skewed and missed at n=5, Gaussian and nailed at n=200")
fig.tight_layout()
save(fig, "laplace_logit")

![Two panels of the Beta posterior (dashed) with the logit-Laplace fit (solid) transformed back to theta. At n=5 the exact posterior is right-skewed and the Laplace curve misses the peak and left shoulder; at n=200 the two curves are indistinguishable.](figures/13-laplace-em-vi/laplace_logit.png)

Laplace also hands you an **evidence approximation** for free — the same Gaussian integral gives $\log p(x) \approx \log p(x,\theta^\star) + \tfrac{d}{2}\log 2\pi - \tfrac12\log|\mathbf{H}|$, where $\mathbf{H}$ is the negative Hessian of the log-joint at the mode and $d$ the dimension. This is how Laplace does model comparison (module 17's evidence, without conjugacy). Check it against the exact Beta-Binomial evidence at $n=100$:

In [ ]:
# 13.2b -- Laplace evidence vs the exact conjugate marginal likelihood, n=100.
a0, b0, s, n = 1.0, 1.0, 30, 100
f = n - s
an, bn = a0 + s, b0 + f                                  # posterior Beta(31, 71)
logpx_exact = np.log(comb(n, s)) + logBeta(an, bn) - logBeta(a0, b0)
th_star = an / (an + bn)                                 # mode of the logit-posterior
log_g = np.log(comb(n, s)) + an * np.log(th_star) + bn * np.log(1 - th_star)   # log joint in phi
H = an * bn / (an + bn)                                  # curvature (1-D)
logpx_lap = log_g + 0.5 * np.log(2 * np.pi) - 0.5 * np.log(H)
print(f"exact   log-evidence = {logpx_exact:.4f}   ({logpx_exact:.2f} to 2 dp)")
print(f"Laplace log-evidence = {logpx_lap:.4f}   ({logpx_lap:.2f} to 2 dp)")

The exact log-evidence is `-4.6151` and Laplace returns `-4.6182` — both `-4.62` to two decimals. At $n=100$ the Gaussian integral prices the evidence to the second decimal without ever doing the Beta-Binomial algebra; that agreement is what licenses Laplace as a model-comparison tool when no conjugate shortcut exists.

## 13.3 EM: coordinate ascent on the ELBO, with a point mass hidden inside

Some models carry **latent variables** $z$ alongside the parameters $\theta$ — cluster labels, missing entries, augmentation variables (module 11). The expectation–maximization (EM) algorithm maximizes $\log p(x\mid\theta)$ over $\theta$ when a direct maximization is blocked by the latent sum $\log p(x\mid\theta) = \log\sum_z p(x,z\mid\theta)$. EM is *also* an ELBO method — the cleanest one — once you write $q$ over $(\theta,z)$.

The minimal example (the full Gaussian mixture, with unknown weights and covariances, is module 19 — we do not build it here): a **two-component 1-D mixture with known, equal variances and known equal weights**, unknown component means $\mu=(\mu_0,\mu_1)$. The latent $z_i\in\{0,1\}$ says which component generated $x_i$. Hold $\theta=\mu$ as a point and let $q(z)=\prod_i q(z_i)$ be a distribution over labels. The ELBO is

$$\mathrm{ELBO}(q,\mu) = \sum_i \sum_{k} q(z_i{=}k)\Big[\log\big(\tfrac12\,\mathcal{N}(x_i;\mu_k,1)\big) - \log q(z_i{=}k)\Big],$$

and EM is *coordinate ascent* on it. **E-step:** maximize over $q$ with $\mu$ fixed — the optimum is the exact conditional $q(z_i{=}k) = p(z_i{=}k\mid x_i,\mu)$ (the "responsibility"), which drives $\mathrm{KL}(q(z)\|p(z\mid x,\mu))$ to zero and makes the ELBO *equal* $\log p(x\mid\mu)$. **M-step:** maximize over $\mu$ with $q$ fixed — a weighted least-squares giving $\mu_k = \sum_i q(z_i{=}k)\,x_i / \sum_i q(z_i{=}k)$. Each half raises the same ELBO, so it never decreases; and because every E-step re-tightens the bound to the true log-likelihood, $\log p(x\mid\mu)$ itself climbs monotonically.

In [ ]:
# 13.3 -- EM for a 2-component 1-D mixture (known sigma=1, known equal weights); unknown means.
mu_true, sig, Nm = np.array([-2.0, 2.0]), 1.0, 200
z_gen = rng.integers(0, 2, size=Nm)
xm = rng.normal(mu_true[z_gen], sig)                 # data from a known-variance mixture

mu = np.array([-0.5, 0.5])                           # deliberately poor init
loglik_trace = []
for _ in range(15):
    # E-step: responsibilities = exact conditional p(z_i | x_i, mu)  (equal weights 0.5)
    l0 = stats.norm(mu[0], sig).logpdf(xm) + np.log(0.5)
    l1 = stats.norm(mu[1], sig).logpdf(xm) + np.log(0.5)
    denom = np.logaddexp(l0, l1)                      # log p(x_i | mu), marginalizing z_i
    loglik_trace.append(denom.sum())                 # incomplete-data log-likelihood
    r1 = np.exp(l1 - denom); r0 = 1 - r1
    # M-step: means are responsibility-weighted averages
    mu = np.array([(r0 * xm).sum() / r0.sum(), (r1 * xm).sum() / r1.sum()])
loglik_trace = np.array(loglik_trace)

print(f"recovered means mu_hat = [{mu[0]:.4f}, {mu[1]:.4f}]   (truth [-2, 2])")
print(f"log-lik: start {loglik_trace[0]:.2f} -> after 1 step {loglik_trace[1]:.2f} "
      f"-> converged {loglik_trace[-1]:.2f}")
print(f"monotone non-decreasing: {bool(np.all(np.diff(loglik_trace) >= -1e-9))}")

From a poor start the means converge to $\hat\mu=$ [`-2.1557`, `1.8968`], and the log-likelihood climbs `-616.52` → `-411.79`, monotone at every step (`True`). Now the reduction the panel insists on, stated precisely:

> **EM is variational inference with a *point mass* on $\theta$ while $q(z)$ is the *exact* conditional $p(z\mid x,\theta)$.** The variational family is $q(\theta,z) = \delta(\theta-\hat\theta)\,\prod_i p(z_i\mid x_i,\hat\theta)$: mean-field between $\theta$ and $z$, but with a degenerate (Dirac) factor on $\theta$ and an *exact* factor on $z$. The asymmetry is the point. EM is honest about the latents — it uses their true conditional — and maximally *dis*honest about the parameters, collapsing all posterior uncertainty in $\theta$ to a single point. That is why EM returns a MAP/ML estimate with no error bars: the $q(\theta)$ it implicitly uses has zero variance by construction.

Full variational inference (next) keeps a genuine distribution on $\theta$ too. EM is the special case that gives it up.

In [ ]:
# figure: monotone log-likelihood, and the fitted mixture density over the data.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(loglik_trace, "o-", color="C1")
ax[0].set(title="EM log-likelihood is monotone (E-step re-tightens the ELBO)",
          xlabel="EM iteration", ylabel=r"$\log p(x\mid\mu)$")
xs = np.linspace(-6, 6, 400)
ax[1].hist(xm, bins=40, density=True, color="C7", alpha=0.5, label="data")
mix = 0.5 * stats.norm(mu[0], sig).pdf(xs) + 0.5 * stats.norm(mu[1], sig).pdf(xs)
ax[1].plot(xs, mix, color="C1", lw=2, label="fitted mixture")
for m in mu_true:
    ax[1].axvline(m, color="k", ls="--", lw=1)
ax[1].set(title="Recovered two-component fit", xlabel="x", ylabel="density")
ax[1].legend()
fig.tight_layout()
save(fig, "em_mixture")

![Left: the EM log-likelihood rising and flattening over 15 iterations, never dropping. Right: a bimodal data histogram with the fitted equal-weight two-Gaussian density overlaid and the true means marked.](figures/13-laplace-em-vi/em_mixture.png)

## 13.4 CAVI on a correlated Gaussian: the variance is too small by exactly $1-\rho^2$

Variational inference keeps a real distribution on everything but restricts $\mathcal{Q}$ to a **mean-field** (fully factorized) family $q(\theta)=\prod_j q_j(\theta_j)$ and maximizes the ELBO by **coordinate ascent variational inference (CAVI)**: cycle through the factors, each update $\log q_j(\theta_j) = \mathbb{E}_{q_{-j}}[\log p(x,\theta)] + \text{const}$ holding the others fixed. It is Gibbs sampling's deterministic cousin — full conditionals replaced by their expected logs — and it inherits Gibbs's enemy, posterior correlation.

Watch it on the most transparent correlated target: a mean-zero bivariate Gaussian $p(\theta)=\mathcal{N}(0,\Sigma)$ with $\Sigma=\begin{psmallmatrix}1&\rho\\\rho&1\end{psmallmatrix}$, correlation $\rho$ *known*. No data, no likelihood — just: how well does a factorized $q(\theta_1)q(\theta_2)$ approximate a correlated Gaussian?

**Setup.** Mean-field forces $q_1\perp q_2$, so it *cannot* represent the tilt of the correlation. **Predict.** It clearly gets the correlation wrong (it reports zero). But will it at least get the **marginals** right — the correct means *and* the correct marginal variances (both $1$)? Commit to a yes/no and, if no, a direction. **Reason.** The intuition: a marginal is a one-variable summary, and mean-field is built from one-variable factors, so surely the factors recover the marginals even if they miss the joint.

**Run.** The CAVI update for a Gaussian is exact and closed-form. With precision $\Lambda=\Sigma^{-1}$, the factor $q_j$ comes out Gaussian with mean $-\Lambda_{jj}^{-1}\sum_{k\ne j}\Lambda_{jk}\mathbb{E}[\theta_k]$ and **variance $1/\Lambda_{jj}$**. At the fixed point the means are all $0$ (correct). But $\Lambda_{jj}=1/(1-\rho^2)$, so the mean-field marginal variance is $1-\rho^2$, against a true marginal variance of $1$.

In [ ]:
# 13.4 -- CAVI on N(0, Sigma) with Sigma = [[1, rho],[rho, 1]]. Marginal variance -> 1 - rho^2.
def cavi_gaussian(rho, iters=50):
    Sigma = np.array([[1.0, rho], [rho, 1.0]])
    Lam = np.linalg.inv(Sigma)                        # precision matrix
    m = np.array([2.0, -2.0])                         # deliberately wrong init means
    for _ in range(iters):                            # coordinate ascent on the means
        m[0] = -(Lam[0, 1] * m[1]) / Lam[0, 0]
        m[1] = -(Lam[1, 0] * m[0]) / Lam[1, 1]
    var_mf = 1.0 / np.diag(Lam)                       # mean-field marginal variances (immediate)
    return m, var_mf, Lam

for rho in (0.8, 0.9):
    m, var_mf, Lam = cavi_gaussian(rho)
    print(f"rho={rho}: rho^2={rho**2:.2f}  1-rho^2={1 - rho**2:.2f}  "
          f"CAVI means={np.round(m, 3)}  CAVI var={var_mf[0]:.4f}  (true var=1)")

# The KL that CAVI actually minimizes, KL(q || p), for rho=0.8:
rho = 0.8
_, var_mf, Lam = cavi_gaussian(rho)
Sigma = np.array([[1.0, rho], [rho, 1.0]])
Q = np.diag(var_mf)
kl_qp = 0.5 * (np.trace(Lam @ Q) - 2 + np.log(np.linalg.det(Sigma) / np.linalg.det(Q)))
print(f"underdispersion factor = 1 - rho^2 = {1 - rho**2:.2f} (exact);  KL(q||p) = {kl_qp:.4f} nats")

**Reconcile.** The means are exact (`0`), but the marginal variance collapses from $1$ to $1-\rho^2 = $ `0.36` at $\rho=0.8$ and `0.19` at $\rho=0.9$ — matching $1-\rho^2$ to the printed digits, not approximately but *exactly*. Mean-field does **not** recover the marginals: it nails their centers and shrinks their spreads by the exact factor $1-\rho^2$. The naive "factors recover marginals" intuition fails because CAVI minimizes $\mathrm{KL}(q\,\|\,p)$ — the *reverse*, mode-seeking KL flagged in module 03. That direction penalizes $q$ for putting mass where $p$ has none, so a factorized $q$ pulls *inward* to fit inside the correlated ellipse rather than spanning it. The result is the signature failure of all mean-field VI: **confident and wrong** — right answer, error bars too tight. Here that under-confidence is a clean multiplicative law you can quote before running anything.

In [ ]:
# figure: the shrunken ellipse. To make "axis-aligned vs tilted" VISIBLE, give the target
# unequal marginal variances (1, 2) — at equal variances the mean-field q is a circle,
# which has no axes to show. The 1-rho^2 law carries over: var_mf_j = (1-rho^2) * Sigma_jj.
from matplotlib.patches import Ellipse
rho = 0.8
Sig_u = np.array([[1.0, rho * np.sqrt(2.0)], [rho * np.sqrt(2.0), 2.0]])   # vars (1, 2), corr 0.8
var_u = 1.0 / np.diag(np.linalg.inv(Sig_u))           # mean-field variances = 1/Lambda_jj
print(f"unequal-variance target diag(Sigma)=(1, 2), rho=0.8: "
      f"mean-field vars = {var_u[0]:.4f}, {var_u[1]:.4f}  (= (1-rho^2) x (1, 2))")
fig, ax = plt.subplots(figsize=(6, 6))
gx = np.linspace(-4.2, 4.2, 200)
XX, YY = np.meshgrid(gx, gx)
pos = np.dstack([XX, YY])
ax.contour(XX, YY, stats.multivariate_normal([0, 0], Sig_u).pdf(pos),
           levels=6, colors="k", linewidths=1)
ax.contour(XX, YY, stats.multivariate_normal([0, 0], np.diag(var_u)).pdf(pos),
           levels=6, colors="C1", linewidths=1)
# 2-sd ellipses for emphasis
for cov, col, lab in [(Sig_u, "k", "true N(0, Σ): tilted, vars (1, 2)"),
                      (np.diag(var_u), "C1", "mean-field q: axis-aligned, vars ×(1−ρ²)")]:
    vals, vecs = np.linalg.eigh(cov)
    ang = np.degrees(np.arctan2(*vecs[:, 1][::-1]))
    e = Ellipse((0, 0), 2 * 2 * np.sqrt(vals[1]), 2 * 2 * np.sqrt(vals[0]),
                angle=ang, fill=False, edgecolor=col, lw=2.5, label=lab)
    ax.add_patch(e)
ax.set(xlim=(-4.2, 4.2), ylim=(-4.2, 4.2), xlabel=r"$\theta_1$", ylabel=r"$\theta_2$",
       title=f"Mean-field q cannot tilt: axis-aligned, each variance ×(1−ρ²)={1 - rho**2:.2f}")
ax.legend(loc="upper left", fontsize=9)
ax.set_aspect("equal")
save(fig, "cavi_ellipse")

![Contours of the true correlated Gaussian (black, visibly tilted ellipse with unequal axes) and the mean-field approximation (orange): an axis-aligned ellipse with the same center, no tilt, inscribed inside the black one and narrower along both coordinate directions.](figures/13-laplace-em-vi/cavi_ellipse.png)

The true ellipse tilts (that *is* the correlation); the mean-field one cannot — independent factors have no off-diagonal term — so it sits axis-aligned and *inscribed*: same center, no tilt, each axis shrunk. The figure's target has unequal marginal variances (1 and 2) precisely so the alignment is visible (at equal variances the mean-field $q$ degenerates to a circle, which has no axes to show), and the $1-\rho^2$ law carries over verbatim: each factor's variance is the matching diagonal entry of $\Sigma$ times $1-\rho^2$, printed as `0.3600` and `0.7200`. The reverse KL is the same `0.5108` nats as the unit-variance case (a diagonal rescaling changes neither the factorized family nor the KL) — the price of independence.

## 13.5 CAVI on the NIG target: grading against module 05's exact answer

The Gaussian toy has a real-data twin: estimating a Normal's mean *and* variance, the Normal-Inverse-Gamma problem of module 05. There the exact posterior couples $\mu$ and $\sigma^2$ (a small $\sigma^2$ makes $\mu$ better-pinned), and the marginal posterior of $\mu$ is a **Student-$t$** with $\nu=2a_n$. Mean-field VI factorizes $q(\mu,\sigma^2)=q(\mu)\,q(1/\sigma^2)$, severing that coupling; the CAVI updates give a Gaussian $q(\mu)$ and a Gamma $q(1/\sigma^2)$. Same setup as module 05: prior parameters $(\mu_0,\kappa_0,a_0,b_0)$, and we compare CAVI's $q(\mu)$ to the exact Student-$t$ marginal.

In [ ]:
# 13.5 -- mean-field VI for the Normal (mean & precision unknown) vs the exact NIG posterior (M05).
rng_nig = np.random.default_rng(3)
xd = rng_nig.normal(5.0, 2.0, size=8)                 # small n so tails matter
mu0, k0, a0, b0 = 0.0, 1.0, 1.0, 1.0                  # weak prior (module 05 convention)
nd = len(xd); xbar = xd.mean(); Sxx = ((xd - xbar)**2).sum()

# exact NIG posterior (module 05's nig_post) and its Student-t marginal for mu:
# (note: an, bn from here on are NIG posterior params -- not §13.1's Beta ones)
kn = k0 + nd
mn = (k0 * mu0 + nd * xbar) / kn
an = a0 + nd / 2
bn = b0 + 0.5 * Sxx + (k0 * nd * (xbar - mu0)**2) / (2 * kn)
exact_mu_sd = np.sqrt(bn / (kn * (an - 1)))           # sd of t_{2an}(loc=mn, ...)

# CAVI: q(mu)=N(mN, 1/lamN), q(tau)=Gamma(aN,bN); coordinate ascent to a fixed point.
Etau = a0 / b0
mN = (k0 * mu0 + xd.sum()) / (k0 + nd)                 # = mn, independent of tau
for _ in range(100):
    v = 1.0 / ((nd + k0) * Etau)                       # variance of q(mu)
    aN = a0 + (nd + 1) / 2
    bN = b0 + 0.5 * (np.sum((xd - mN)**2) + nd * v + k0 * ((mN - mu0)**2 + v))
    Etau = aN / bN
cavi_mu_sd = np.sqrt(v)

print(f"exact  posterior mu: mean {mn:.4f}  sd {exact_mu_sd:.4f}  (Student-t, df={2 * an:.0f})")
print(f"CAVI   q(mu):        mean {mN:.4f}  sd {cavi_mu_sd:.4f}  (Gaussian)")
print(f"mean error |mN - mn| = {abs(mN - mn):.2e};  sd ratio CAVI/exact = {cavi_mu_sd / exact_mu_sd:.4f}")
print(f"closed form sqrt((an-1)/an) = {np.sqrt((an - 1) / an):.4f}   (CAVI var = the t's squared scale)")

# figure: CAVI q(mu) (Gaussian) vs the exact Student-t marginal posterior of mu.
fig, ax = plt.subplots()
mg = np.linspace(mn - 4 * exact_mu_sd, mn + 4 * exact_mu_sd, 400)
t_scale = np.sqrt(bn / (an * kn))                     # scale of t_{2an}(loc=mn)
ax.plot(mg, stats.t(df=2 * an, loc=mn, scale=t_scale).pdf(mg), "k--", lw=2,
        label=f"exact posterior  $t_{{{2 * an:.0f}}}$ (sd {exact_mu_sd:.3f})")
ax.plot(mg, stats.norm(mN, cavi_mu_sd).pdf(mg), color="C1", lw=2,
        label=f"CAVI  $q(\\mu)$ Gaussian (sd {cavi_mu_sd:.3f})")
ax.axvline(mn, color="C0", ls=":", lw=1, label="shared mean")
ax.set(xlabel=r"$\mu$", ylabel="density",
       title="CAVI matches the mean, misses the tails: Gaussian narrower than the exact t")
ax.legend()
save(fig, "cavi_nig")

CAVI's posterior mean for $\mu$ is `3.6479`, identical to the exact $m_n=$ `3.6479` to machine precision — mean spot-on, as in the toy. Its standard deviation is `0.8998` against the exact `1.0060`, a ratio of `0.8944`: the variational $\mu$-posterior is about 11% too narrow. And that ratio is not an empirical accident — it equals $\sqrt{(a_n-1)/a_n}=\sqrt{0.8}=$ `0.8944` exactly: CAVI's Gaussian variance lands on the exact $t$'s squared *scale* $b_n/(a_n\kappa_n)$, missing precisely the $\nu/(\nu-2)$ tail-inflation factor that integrating over $\sigma^2$ contributes. The mechanism, in one sentence: the Gaussian $q(\mu)$ conditions on $\mathbb{E}[1/\sigma^2]$ where the exact Student-$t$ *integrates* over an uncertain $\sigma^2$. The df is `10`; the smaller the $n$, the heavier the true $t$ and the worse the deficit.

![The exact Student-t marginal posterior for mu (dashed) and the CAVI Gaussian (solid) sharing a peak at 3.65; the Gaussian is taller and narrower, with the dashed t showing visibly heavier tails on both sides.](figures/13-laplace-em-vi/cavi_nig.png)

## 13.6 SVI and AutoNormal: automatic VI against NUTS

Hand-deriving CAVI updates does not scale. Modern VI does **stochastic variational inference (SVI)**: pick a flexible guide $q_\phi$, and maximize a Monte-Carlo estimate of the ELBO by stochastic gradient ascent on $\phi$ (the reparameterization trick makes $\nabla_\phi \mathrm{ELBO}$ an expectation you can sample). NumPyro's `AutoNormal` guide is a mean-field Gaussian in the model's unconstrained space — this is ADVI (automatic differentiation VI). We compare it to NUTS (module 12) on the same model, copying the idioms from `tools/ppl_idioms.py` verbatim.

**Setup.** SVI's `AutoNormal` is mean-field Gaussian; NUTS is the asymptotically-exact reference. **Predict.** Section 13.4's law says where each will land: commit to which of {posterior *means*, posterior *interval widths*} agree between the two — and then to a number: SVI's $\mathrm{sd}(\sigma)$ as a fraction of NUTS's will be **>0.9**, **≈0.7**, or **<0.5**. Pick one. **Reason.** If mean-field is "means right, variances too small," the point estimates should match and SVI's intervals should be narrower — but with only two, mildly coupled parameters, most guts say the deficit is cosmetic: >0.9.

In [ ]:
# 13.6 -- SVI (AutoNormal / ADVI) vs NUTS on one model. Idioms copied from tools/ppl_idioms.py.
import time
import jax
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoNormal

y_obs = rng.normal(1.0, 2.0, size=50)                 # numpy array (idiom: obs stay numpy)
N_OBS = 50

def normal_model(N, y=None):
    mu = numpyro.sample("mu", dist.Normal(0.0, 10.0))
    sigma = numpyro.sample("sigma", dist.HalfNormal(5.0))
    with numpyro.plate("N", N):
        numpyro.sample("y", dist.Normal(mu, sigma), obs=y)

t0 = time.perf_counter()
mcmc = MCMC(NUTS(normal_model), num_warmup=800, num_samples=800, num_chains=2,
            chain_method="sequential", progress_bar=False)
mcmc.run(jax.random.PRNGKey(SEED), N=N_OBS, y=y_obs)
t_nuts = time.perf_counter() - t0
ps = mcmc.get_samples()
mu_n, sig_n = np.asarray(ps["mu"]), np.asarray(ps["sigma"])

t0 = time.perf_counter()
guide = AutoNormal(normal_model)
svi = SVI(normal_model, guide, numpyro.optim.Adam(0.05), Trace_ELBO())
res = svi.run(jax.random.PRNGKey(SEED), 4000, N=N_OBS, y=y_obs, progress_bar=False)
qpost = guide.sample_posterior(jax.random.PRNGKey(3), res.params, sample_shape=(4000,))
t_svi = time.perf_counter() - t0
mu_s, sig_s = np.asarray(qpost["mu"]), np.asarray(qpost["sigma"])

print(f"means agree:   NUTS mu {mu_n.mean():.3f} / SVI mu {mu_s.mean():.3f};"
      f"  NUTS sigma {sig_n.mean():.3f} / SVI sigma {sig_s.mean():.3f}")
print(f"sd(mu):    NUTS {mu_n.std():.3f}  SVI {mu_s.std():.3f}")
print(f"sd(sigma): NUTS {sig_n.std():.3f}  SVI {sig_s.std():.3f}   ratio SVI/NUTS {sig_s.std() / sig_n.std():.3f}")
print(f"sigma 97.5% tail: NUTS {np.quantile(sig_n, 0.975):.3f}  SVI {np.quantile(sig_s, 0.975):.3f}")
print(f"final ELBO loss = {float(res.losses[-1]):.1f}")

# figure: posterior for sigma (tails light) + the wall-clock trade (timing lives only here).
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(sig_n, bins=50, density=True, color="C0", alpha=0.5, label="NUTS (reference)")
ax[0].hist(sig_s, bins=50, density=True, color="C1", alpha=0.5, label="SVI / AutoNormal")
ax[0].axvline(np.quantile(sig_n, 0.975), color="C0", ls="--", lw=1)
ax[0].axvline(np.quantile(sig_s, 0.975), color="C1", ls="--", lw=1)
ax[0].set(title="Posterior for σ: SVI centered right, right tail clipped",
          xlabel=r"$\sigma$", ylabel="density")
ax[0].legend()
ax[1].bar(["NUTS", "SVI"], [t_nuts, t_svi], color=["C0", "C1"])
ax[1].set(title="Wall-clock: the speed you buy with the tails", ylabel="seconds")
for i, t in enumerate([t_nuts, t_svi]):
    ax[1].annotate(f"{t:.1f}s", (i, t), textcoords="offset points", xytext=(0, 3), ha="center")
fig.tight_layout()
save(fig, "svi_vs_nuts")

**Reconcile.** The point estimates agree: NUTS gives posterior means $\mu=$ `0.776`, $\sigma=$ `1.905`; SVI gives `0.739` and `1.973` — close enough to make identical decisions. The spreads do not agree, in the predicted direction. SVI's $\sigma$-posterior has standard deviation `0.146` against NUTS's `0.209`, a ratio of `0.700` — the middle option, and a long way from the ">0.9, it's cosmetic" guess — and its upper tail is clipped: the $\sigma$ 97.5% quantile is `2.282` for SVI versus `2.366` for NUTS. Mean-field again: right center, tails too light, because `AutoNormal` factorizes over $(\mu,\sigma)$ and reports Gaussian tails in the unconstrained space. The wall-clock is the compensation — SVI finishes in a fraction of NUTS's time (figure), converging to a final ELBO loss of `105.6`. That is the whole trade in one screen: an order of magnitude faster, point estimates you can trust, interval widths you cannot.

![Left: overlaid posterior histograms for sigma from NUTS and SVI, matched at the center but with the SVI histogram narrower and its 97.5% line inside NUTS's. Right: a two-bar wall-clock chart with NUTS much taller than SVI.](figures/13-laplace-em-vi/svi_vs_nuts.png)

> **When to trust VI.** *Trust it for:* point predictions, posterior means, MAP-adjacent estimates, and learned representations (embeddings) — anything that depends on *where* the posterior sits. Mean-field nails the center. *Do not trust it for:* interval widths, tail probabilities, tail risk, calibration, or any decision priced by the *edges* of the posterior — VI reports these too small by a factor it will not tell you (here $1-\rho^2$, in general unknown). If you need honest uncertainty, VI gives you a fast, biased-narrow starting point; verify the widths against NUTS or importance sampling before you bet on them. This is exactly the ledger you will carry into the **variational autoencoder** (module 25), whose training loss *is* the ELBO of this module with an amortized, neural-network guide $q_\phi(z\mid x)$ — same objective, same reverse-KL, same under-dispersion, now over a latent code.

## Bridge — one objective, three classical algorithms, and the ML present

The ELBO is the hinge between this course's two halves. Behind it, **KL to the posterior** (module 03's divergence, now steering an approximation rather than scoring a model) unifies three procedures that predate the unification by decades: Laplace (1774), EM (Dempster–Laird–Rubin 1977), and variational methods (1990s). Ahead of it, the same $\mathrm{ELBO} = \mathbb{E}_q[\log p(x\mid z)] - \mathrm{KL}(q(z)\,\|\,p(z))$ — reconstruction minus a KL-to-the-prior regularizer — is the objective of every VAE and the workhorse of amortized inference (module 25). The reduction chain is worth memorizing: **EM = VI with a point mass on $\theta$; Laplace = the Gaussian matching the posterior's peak curvature (not the ELBO-optimal Gaussian, except asymptotically); mean-field VI = the factorized ELBO maximizer, under-dispersed by construction.** All three are line 2 of the spine — conditioning — done approximately, and each fails in a *signed, predictable* way, which is the only kind of failure a practitioner can plan around.

## Pitfalls

- **KL to the prior instead of the posterior.** The identity is $\log p(x)=\mathrm{ELBO}(q)+\mathrm{KL}(q\,\|\,p(\cdot\mid x))$. Maximizing the ELBO minimizes the distance to the *posterior*. (The VAE's ELBO *also* contains a separate $\mathrm{KL}(q(z)\|p(z))$ term to the *prior* — a regularizer, a different object. Do not conflate them.)
- **Laplace on a constrained parameter.** A Gaussian on $[0,1]$ or $(0,\infty)$ leaks mass past the boundary (`0.1318` below zero in §13.2) and is skewed-wrong at small $n$. Reparameterize to an unconstrained scale (logit, log) first — the same move as parameterizing a network's variance in log-space.
- **Reading VI error bars as real.** Mean-field VI is *confidently* wrong: means right, variances shrunk by $1-\rho^2$ (exactly, for a correlated Gaussian; unpredictably, in general). Reporting a variational credible interval as if it were calibrated is the field's most common VI abuse.
- **Trusting EM's "estimate" as a posterior.** EM's $q(\theta)$ is a Dirac spike; it has *no* uncertainty by construction. It answers "what $\theta$ maximizes the likelihood," never "how sure am I about $\theta$."
- **Declaring VI converged from a flat ELBO.** A converged ELBO means you found the best member of $\mathcal{Q}$ — it says nothing about how far that best member is from the true posterior. The gap is the KL you cannot see without a second method. A flat ELBO on a bad family is a confident mirage.

## Exercises

**Exercise 13.1 — Which mode does reverse-KL keep? (surprising)**
*Setup:* The target is a well-separated bimodal mixture $p = \tfrac12\mathcal{N}(-4,1)+\tfrac12\mathcal{N}(4,1)$. You approximate it by a *single* Gaussian $q=\mathcal{N}(m,v)$, once by minimizing the reverse KL $\mathrm{KL}(q\|p)$ (what VI does — tried from two starting points) and once by the forward $\mathrm{KL}(p\|q)$ (what moment-matching / expectation propagation does).
*Predict:* Where do the optima land? Give $m$ and a rough $\text{sd}$ for each. Which objective straddles both modes, and which picks one?
*Reason:* Both are "the best Gaussian," so the naive guess is that both center at $0$ and differ only slightly in width.
*Run:*

In [ ]:
xs = np.linspace(-12, 12, 6001); dx = xs[1] - xs[0]
p_mix = 0.5 * stats.norm(-4, 1).pdf(xs) + 0.5 * stats.norm(4, 1).pdf(xs)
def kl_qp(par):                                       # reverse KL(q||p): what VI minimizes
    q = stats.norm(par[0], np.exp(par[1])).pdf(xs) + 1e-300
    return np.sum(q * np.log(q / (p_mix + 1e-300))) * dx
from scipy.optimize import minimize
rev_mode = minimize(kl_qp, [4.0, 0.0], method="Nelder-Mead")   # init near one mode
rev_mid = minimize(kl_qp, [0.0, 0.5], method="Nelder-Mead")    # init between the modes
m_fwd = np.sum(xs * p_mix) * dx
sd_fwd = np.sqrt(np.sum(xs**2 * p_mix) * dx - m_fwd**2)        # forward KL: moment match
print(f"reverse KL, mode init:     m={rev_mode.x[0]:.3f}  sd={np.exp(rev_mode.x[1]):.3f}  KL={rev_mode.fun:.3f}")
print(f"reverse KL, centered init: m={rev_mid.x[0]:.3f}  sd={np.exp(rev_mid.x[1]):.3f}  KL={rev_mid.fun:.3f}")
print(f"forward KL (moment match): m={m_fwd:.3f}  sd={sd_fwd:.3f}")

<details><summary>Reconcile</summary>

The reverse-KL *global* optimum collapses onto a single bump: $m=$ `4.000`, $\text{sd}=$ `1.001`, with $\mathrm{KL}=$ `0.693` $=\ln 2$ — exactly the log of the mixture weight it abandoned (inside one bump, $\log q - \log p \approx \log q - \log(\tfrac12 q) = \ln 2$). Forward KL centers at $m=$ `0.000` with $\text{sd}=$ `4.123` $=\sqrt{17}$, spanning both modes. The naive "both center at 0" is half-wrong: only the forward KL does. Reverse KL is **mode-seeking** — it pays $\mathbb{E}_q[\log q/p]$, which explodes if $q$ puts mass in the near-empty valley between the modes, so it hides inside one of them; forward KL is **mode-covering** — penalized for low $q$ wherever $p$ has mass, so it stretches. The centered init is the second lesson: it converges to a *worse* stationary point (KL `1.851`, a valley-straddling Gaussian) — the reverse objective has local optima, so VI can hand you a different answer per initialization. This is the mechanism behind every under-dispersion result in §13.4–13.6, sharpened to total mode loss: on a genuinely multimodal posterior, VI doesn't just narrow the posterior — it can silently discard entire modes.
</details>

**Exercise 13.2 — The underdispersion factor at high correlation.**
*Setup:* Mean-field CAVI on the correlated Gaussian of §13.4, now at $\rho=0.95$.
*Predict:* By what factor does the mean-field marginal variance undershoot the true variance of $1$? Give a number.
*Reason:* At $\rho=0.8$ the factor was $1-\rho^2=0.36$; extrapolating "linearly" from there suggests something like $0.30$ at $\rho=0.95$.
*Run:*

In [ ]:
for rho in (0.8, 0.9, 0.95, 0.99):
    print(f"rho={rho}: mean-field variance = 1 - rho^2 = {1 - rho**2:.4f}")

<details><summary>Reconcile</summary>

The factor is $1-0.95^2=$ `0.0975` — the variational marginal reports about a *tenth* of the true variance, so its standard deviation is off by $\sqrt{0.0975}\approx0.31$, more than 3× too narrow. The linear extrapolation badly underestimates the damage because $1-\rho^2$ falls off *quadratically* near $\rho=1$: at $\rho=0.99$ it is `0.0199`, a 50× variance collapse. The lesson: mean-field VI is not uniformly a little overconfident — it is *catastrophically* overconfident exactly when the posterior is most correlated, which is exactly when you most need to know the correlation. High posterior correlation is mean-field's worst case, and it arrives silently (the ELBO looks fine).
</details>

**Exercise 13.3 — Laplace evidence buys automatic Occam (ML/stats bridge).**
*Setup:* Two models for a coin's 10 flips (6 heads): $M_1$ fixes $\theta=0.5$ (no free parameter); $M_2$ puts $\theta\sim\mathrm{Beta}(1,1)$ and estimates it. You compare them by evidence $p(x\mid M)$ — the quantity Laplace approximates and module 17 makes central.
*Predict:* The flexible $M_2$ fits the data at least as well as $M_1$ at its best $\theta$. Does the *evidence* therefore favor $M_2$?
*Reason:* "More parameters fit better," the overfitting reflex from ML model selection.
*Run:*

In [ ]:
from scipy.special import comb as C, betaln
s, n = 6, 10
ev_fixed = C(n, s) * 0.5**n                            # p(x | M1): theta pinned at 0.5
ev_free = C(n, s) * np.exp(betaln(1 + s, 1 + n - s) - betaln(1, 1))   # p(x | M2): Beta(1,1)
print(f"evidence M1 (theta=0.5)      = {ev_fixed:.4f}")
print(f"evidence M2 (theta free)     = {ev_free:.4f}")
print(f"Bayes factor M1/M2           = {ev_fixed / ev_free:.3f}")

<details><summary>Reconcile</summary>

The evidence *favors the simpler* $M_1$: `0.2051` versus `0.0909`, a Bayes factor of `2.256` in $M_1$'s favor — even though $M_2$ can fit the data better at its best $\theta=0.6$. Evidence is the *prior-predictive* $p(x\mid M)=\int p(x\mid\theta)p(\theta)\,d\theta$: $M_2$ must spread its prior belief over all $\theta\in[0,1]$, so it spends probability on data it did *not* see, and the average likelihood pays an automatic complexity tax — the **Occam factor** that Laplace's $-\tfrac12\log|\mathbf H|$ term encodes and module 17 (signature S5) derives in full. Six-of-ten is unremarkable for a fair coin, so paying for a free parameter is not worth it. This is why Bayesian evidence penalizes complexity *without* a held-out set or an explicit regularizer — the integral does the CV for you.
</details>

**Exercise 13.4 — Does more optimization fix the underdispersion?**
*Setup:* In §13.4, mean-field CAVI at $\rho=0.9$ reported marginal variance $0.19$ instead of $1$. A colleague argues coordinate ascent simply hadn't converged — run 10,000 sweeps instead of 2 and the variance will fill in. Alternatively, you could keep 2 sweeps but *change the family* to a full-covariance Gaussian $q$.
*Predict:* Which intervention recovers the true variance — 5,000× more sweeps, the family change, both, or neither?
*Reason:* "Under-dispersion is an optimization artifact — iterate longer and it closes." (The reflex from training loss curves: more epochs, better fit.)
*Run:*

In [ ]:
for iters in (2, 10_000):
    m_ex, var_ex, _ = cavi_gaussian(0.9, iters=iters)
    print(f"CAVI sweeps = {iters:>6}: mean-field marginal variance = {var_ex[0]:.4f}")
# change the FAMILY instead: the best full-covariance Gaussian q IS the target
Sig9 = np.array([[1.0, 0.9], [0.9, 1.0]])
Lam9 = np.linalg.inv(Sig9)
kl_full = 0.5 * (np.trace(Lam9 @ Sig9) - 2 + np.log(np.linalg.det(Sig9) / np.linalg.det(Sig9)))
print(f"full-covariance q = Sigma: marginal variance = {Sig9[0, 0]:.4f}, KL = {kl_full:.4f}")

<details><summary>Reconcile</summary>

Only the family change works. The mean-field variance is `0.1900` at 2 sweeps and `0.1900` at 10,000 — bit-identical, because each factor's variance is $1/\Lambda_{jj}=1-\rho^2$ from the *first* update; iteration count only moves the means. The under-dispersion is **not** an optimization failure; it is the *exact optimum* of the reverse KL on the factorized family. A full-covariance Gaussian $q$, by contrast, contains the target itself, so its optimum is exact: variance `1.0000`, $\mathrm{KL}=$ `0.0000`. This is the last pitfall made concrete: a flat, converged ELBO certifies you found the best member of $\mathcal{Q}$ and says nothing about the gap to the truth. To widen VI's tails you must change the *family* — in NumPyro, swap `AutoNormal` for `AutoMultivariateNormal` or a normalizing-flow guide — never the step count. (The same holds for §13.6's SVI: more Adam steps polish the wrong optimum.)
</details>

## Takeaways

- **One identity governs all three methods:** $\log p(x) = \mathrm{ELBO}(q) + \mathrm{KL}(q\,\|\,p(\cdot\mid x))$ — KL to the *posterior*. Since $\log p(x)$ is fixed, maximizing the ELBO is minimizing that KL, computable without the intractable evidence.
- **Three methods = three choices of $q$'s family:** Laplace (Gaussian at the peak, on an unconstrained scale), EM (point mass on $\theta$, exact conditional on $z$), mean-field VI (factorized, coordinate-ascended). Same objective, different signed failure.
- **Laplace** is cheap and asymptotically justified (BvM) but degrades at small $n$ (KL `0.0264` at $n{=}5$ vs `0.0022` at $n{=}200$) and must run on unconstrained coordinates; its evidence approximation matches the exact conjugate marginal to 2 dp at $n{=}100$ (`-4.62`).
- **EM is VI with a Dirac $q(\theta)$ and exact $q(z)$** — the asymmetry is why it climbs the likelihood monotonically yet returns no uncertainty on $\theta$.
- **Mean-field's signature failure is exact:** on a correlated Gaussian the means are right and the marginal variances are shrunk by $1-\rho^2$ (`0.36` at $\rho{=}0.8$), because CAVI minimizes the mode-seeking *reverse* KL. On the NIG target the same effect makes $q(\mu)$'s sd `0.8998` vs the exact `1.0060`.
- **SVI/ADVI trades tails for speed:** point estimates and means match NUTS; interval widths run narrow (SVI $\mathrm{sd}(\sigma)$ `0.700`× NUTS) — and no amount of extra optimization fixes it, because it is the true optimum of the wrong family.
- **When to trust VI:** yes for point predictions and embeddings; no for tail risk and interval widths. The VAE (module 25) is this module's ELBO with an amortized neural guide — same objective, same under-dispersion.